In [ ]:
# Toda célula de um caderno marimo é uma função Python; o que ela devolve no "return" fica para as outras células.
import marimo as mo

In [ ]:
# O PAINEL: a história em português simples e, numa caixa fechada, os detalhes técnicos (lê o PIPELINE.md).
import sys as _sys
_pasta = str(mo.notebook_dir())
if _pasta not in _sys.path:
    _sys.path.insert(0, _pasta)
import painel
painel.mostrar(mo)

# Ciclo 2, etapa 1: conhecer o material de novo

**Em português:** no ciclo 1 a gente já olhou esta pilha de exemplos. Por que olhar de novo? Porque agora ela é
maior (a prova do ciclo 1, já aberta, pode virar material de estudo) e porque o ciclo 2 quer atacar erros
específicos. Antes de construir qualquer coisa contra um erro, a gente **conta** quantas vezes ele aparece: se for
raro, não vale a pena (regra 15 do repositório).

| O que olhamos | Em português | Em termos técnicos |
|---|---|---|
| **A pilha de estudo** | rodadas 1 a 3, sem nenhuma linha da rodada 4 (a prova nova, lacrada) | tabela de desenvolvimento, conferência do lacre |
| **"Mesmo assunto, outro capítulo"** | quantas fontes são da mesma história mas de outro fato, e parecem muito com a notícia | pares "contexto relacionado" com cosseno alto |
| **A idade da fonte** | em que faixa de dias esses casos aparecem | taxa por faixa de idade |
| **Os rótulos duvidosos** | quantos rótulos saíram do desempate de três juízes | origem do rótulo |
| **O app mudou entre as rodadas?** | a rodada 1 foi capturada com o codificador antigo; as respostas da regra mudam? | mudança de população nas entradas categóricas |
| **O raio-x oficial** | a ficha oficial do dado | `guard.profile` |

**O que eu espero, escrito antes de rodar:**

- 10.294 pares (8.073 do estudo do ciclo 1 mais 2.221 da prova dele) e 12 a 13 "mesmo" em cada 100. É conta, não
  palpite: os dois números já foram medidos no ciclo 1.
- "Contexto relacionado" que se parece com "mesmo fato" (cosseno pelo menos na mediana do cosseno dos "mesmo"):
  entre 10 e 25 de cada 100 "contexto relacionado", ou seja, de 180 a 450 pares. Bem acima do mínimo de 5 casos.
- Mais da metade desses casos com a fonte entre 1 e 30 dias, onde o ciclo 1 errou mais.
- Os de maioria de três: 226, com "mesmo" em 72 de cada 100 (já medido em 07/10).
- A regra diz "assunto equivalente pela comparação de sentido" numa proporção diferente na rodada 1 e nas 2 e 3, com
  diferença maior que 2 pontos entre os pares em que ela comparou. É hipótese: pode dar nulo.

Nada aqui ajusta nada: é só leitura e contagem.

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# O caminho até src/ do challenge; .parent sobe uma pasta (de ml_pipeline/ para a raiz do repositório).
SRC = mo.notebook_dir().parent / "challenges" / "challenge-01-desinformacao-v2" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import prepare

# A tabela com tudo o que o app sabe de cada par (etapa 5 do ciclo 1), mais o cosseno das entradas (etapa 7).
tabela = prepare.tabela_do_decisor()                                   # uma linha por par, indexada pelo id
tabela = tabela.join(prepare.entradas_do_decisor()[["cosseno", "sobreposicao"]])

# A TRAVA DO LACRE: nenhuma matéria da rodada 4 pode estar aqui.
_lacre = json.loads((SRC.parent / "data" / "interim" / "rotulagem-4" / "lacre.json").read_text(encoding="utf-8"))
assert set(tabela["rodada"]) == {1, 2, 3}, f"rodadas inesperadas: {sorted(set(tabela['rodada']))}"
assert not set(tabela["materia"]) & set(_lacre["teste"]), "linha da rodada 4 (lacrada) na pilha de estudo"
f"{len(tabela):,} pares das rodadas 1 a 3; nenhuma das {len(_lacre['teste'])} matérias lacradas da rodada 4".replace(",", ".")

In [ ]:
# A PILHA, por rodada e pelo lado em que cada par estava no ciclo 1 (estudo ou prova).
pilha = (tabela.groupby(["rodada", "lado"])
         .agg(pares=("mesmo", "size"), materias=("materia", "nunique"), mesmo=("mesmo", "mean"))
         .assign(**{"mesmo de cada 100": lambda d: (100 * d["mesmo"]).round(1)}).drop(columns="mesmo"))
_total = pd.DataFrame({"pares": [len(tabela)], "materias": [tabela["materia"].nunique()],
                       "mesmo de cada 100": [round(100 * tabela["mesmo"].mean(), 1)]},
                      index=pd.MultiIndex.from_tuples([("todas", "")], names=["rodada", "lado"]))
pd.concat([pilha, _total])

In [ ]:
# DE ONDE VEIO CADA RÓTULO: consenso dos dois juízes, consenso só no binário, desempate de três, decisão do Matheus.
origem = (tabela.groupby("origem").agg(pares=("mesmo", "size"), mesmo=("mesmo", "mean"))
          .assign(**{"mesmo de cada 100": lambda d: (100 * d["mesmo"]).round(1)}).drop(columns="mesmo"))
origem

In [ ]:
# "MESMO ASSUNTO, OUTRO CAPÍTULO": fontes rotuladas "contexto relacionado" (o mesmo assunto, outro fato) que se
# parecem com a notícia tanto quanto uma fonte típica do mesmo fato. A régua do "parecer": a mediana do cosseno dos
# pares "mesmo". É o erro mais caro do ciclo 1 (etapa 13): mostrar como confirmação o que é outro capítulo.
regua_cosseno = float(tabela.loc[tabela["mesmo"] == True, "cosseno"].median())   # noqa: E712
contexto = tabela[tabela["relacao"] == "contexto_relacionado"]
parecidos = contexto[contexto["cosseno"] >= regua_cosseno]

# Faixas de idade da fonte (dias entre a fonte e a matéria), as mesmas da etapa 13 do ciclo 1.
def faixa(dias):
    if pd.isna(dias):
        return "sem data"
    if dias <= 1:
        return "mesmo dia ou véspera"
    if dias <= 30:
        return "2 a 30 dias"
    return "mais de um mês"

_faixas = tabela["idade_dias"].map(faixa)
por_idade = pd.DataFrame({
    "pares": _faixas.value_counts(),
    "mesmo de cada 100": (100 * tabela.groupby(_faixas)["mesmo"].mean()).round(1),
    "contexto parecido": parecidos["idade_dias"].map(faixa).value_counts(),
}).fillna(0).astype({"pares": int, "contexto parecido": int})
continuacoes = {
    "régua do 'parecer' (mediana do cosseno dos 'mesmo')": round(regua_cosseno, 3),
    "pares 'contexto relacionado'": len(contexto),
    "dos quais parecem 'mesmo' (cosseno na régua ou acima)": len(parecidos),
    "de cada 100 'contexto relacionado'": round(100 * len(parecidos) / max(len(contexto), 1), 1),
    "matérias em que aparecem": int(parecidos["materia"].nunique()),
    "com a fonte de 1 a 30 dias (de cada 100)": round(100 * np.mean(parecidos["idade_dias"].between(1, 30)), 1),
}
continuacoes

In [ ]:
por_idade

**Correção da contagem acima (escrita depois de rodar, sem apagar a anterior).** Contei só os "contexto relacionado"
(C), e a expectativa errou: 57, não 180 a 450. Mas a `REGRA.md` manda rotular como **outro acontecimento** (D) "o
mesmo tipo de fato em outra data" e "outra época ou edição". Ou seja, boa parte das "histórias que continuam" é D, e
a contagem certa é **todo par que não é o mesmo fato e se parece com um** (cosseno na régua ou acima), seja qual for a
letra. É a contagem da célula abaixo.

In [ ]:
# A CONTAGEM CORRIGIDA: os "parecidos que não são o mesmo fato", em qualquer letra (C, D ou E).
parecidos_nao_mesmo = tabela[(tabela["mesmo"] == False) & (tabela["cosseno"] >= regua_cosseno)]   # noqa: E712
_mesmo_acima = int(((tabela["mesmo"] == True) & (tabela["cosseno"] >= regua_cosseno)).sum())         # noqa: E712
_faixa = parecidos_nao_mesmo["idade_dias"].map(
    lambda d: "sem data" if pd.isna(d) else "mesmo dia ou véspera" if d <= 1 else "2 a 30 dias" if d <= 30 else "mais de um mês")
{
    "pares que não são o mesmo fato e parecem um": len(parecidos_nao_mesmo),
    "matérias em que aparecem": int(parecidos_nao_mesmo["materia"].nunique()),
    "por letra": parecidos_nao_mesmo["relacao"].fillna("sem letra (consenso só no binário)").value_counts().to_dict(),
    "por idade da fonte": _faixa.value_counts().to_dict(),
    "para comparar: pares 'mesmo' acima da mesma régua": _mesmo_acima,
}

In [ ]:
# O APP MUDOU ENTRE AS RODADAS? A rodada 1 são capturas do grupo, com o codificador antigo e os cortes 0,72/0,55 da
# regra; as rodadas 2 e 3, o lume-mpnet-ajustado com 0,62/0,35. "assunto = ia_equivalente" é justamente o assunto
# decidido pela comparação de sentido, que depende desses cortes. Só entre os pares em que a regra comparou.
_comparou = tabela[tabela["comparou"] == 1]
mudanca_do_app = pd.DataFrame({
    "pares em que a regra comparou": _comparou.groupby("rodada").size(),
    "de cada 100 pares, a regra comparou": (100 * tabela.groupby("rodada")["comparou"].mean()).round(1),
    "assunto equivalente pelo sentido (de cada 100 comparados)":
        (100 * _comparou.groupby("rodada")["assunto"].apply(lambda s: (s == "ia_equivalente").mean())).round(1),
    "assunto em comum por palavras (de cada 100 comparados)":
        (100 * _comparou.groupby("rodada")["assunto"].apply(lambda s: (s == "em_comum").mean())).round(1),
    "regra diz 'mesmo' (de cada 100 pares)":
        (100 * tabela.groupby("rodada")["regra"].apply(lambda s: s.isin(["mesmo_acontecimento", "detalhe_divergente"]).mean())).round(1),
})
mudanca_do_app

## O raio-x oficial (`guard.profile`)

Grava `ml_pipeline/data_profile.json`, que o painel, as figuras da etapa 2 e as travas do pipeline leem. As mesmas
escolhas do ciclo 1: alvo `mesmo` (o binário que o produto decide), grupo `grupo` (o acontecimento: pares do mesmo
fato nunca se separam entre estudo e simulado). O raio-x roda sobre a tabela inteira, com as colunas de rótulo e de
processo: por isso ele acusa `relacao` como suspeita de vazamento, e é para acusar mesmo (ela É o rótulo, em cinco
letras). Nenhuma delas entra no modelo (lista `PROIBIDAS` do `prepare.py`).

In [ ]:
import guard

# As colunas com texto longo ou listas (trecho, página, detalhes) não ajudam o raio-x e só o deixam lento.
_para_o_raio_x = tabela.drop(columns=["trecho", "pagina", "detalhes_acontecimento", "trecho_lido"])
perfil = guard.profile(_para_o_raio_x, target="mesmo", group_col="grupo", state_dir=mo.notebook_dir())
{k: perfil[k] for k in ("traits", "target", "leakage_suspects") if k in perfil}

# Ciclo 2, etapa 2: desenhar o material

**Em português:** a etapa 1 contou; esta desenha. Duas perguntas viram figura, porque desenhadas elas decidem melhor
do que em tabela: **a rodada 1 é mesmo outro mundo?** e **onde ficam as fontes que enganam** (as que se parecem com a
notícia mas são outro fato)? Antes, as quatro figuras que o pipeline exige.

| O que olhamos | Em português | Em termos técnicos |
|---|---|---|
| **As figuras do pipeline** | buracos, equilíbrio das classes, distribuições, correlações | `guard.eda_figures` |
| **A rodada 1, lado a lado** | por rodada: quanto a regra comparou, quanto ela diz "mesmo", quanto é "mesmo" de verdade | mudança de população |
| **As fontes que enganam** | cada fonte num mapa: semelhança com a notícia contra a idade dela | dispersão cosseno × idade, por classe |

**O que eu espero, escrito antes de rodar.** A figura da rodada 1 só desenha números já medidos na etapa 1 (a regra
comparou em 94,5% dos pares da rodada 1, contra 32,1 e 24,1). A das fontes que enganam mede uma coisa nova: entre as
fontes que se parecem com a notícia (cosseno na régua de 0,74 ou acima), quantas de cada 100 são de fato o mesmo
fato, por idade. Espero mais de 90 no mesmo dia ou véspera, de 60 a 85 de 2 a 30 dias, menos de 40 acima de um mês e
menos de 50 sem data. Se acima de um mês der mais de 40, a idade separa menos do que a etapa 1 sugeriu.

In [ ]:
# As figuras obrigatórias do pipeline, na mesma tabela do raio-x (sem as colunas de texto longo).
_para_figuras = tabela.drop(columns=["trecho", "pagina", "detalhes_acontecimento", "trecho_lido"])
figuras_obrigatorias = guard.eda_figures(_para_figuras, target="mesmo", state_dir=mo.notebook_dir())

# A nossa leitura de cada uma, com números calculados agora (a automática, em inglês, vai para o PIPELINE.md).
_maioria = (tabela["mesmo"] == False).mean()                                     # noqa: E712
_explicacoes = {
    "01_missingness.png": "**Vazios por coluna (do pipeline).** O vazio que mais pesa é a data da fonte (`idade_dias`): "
        f"{tabela['idade_dias'].isna().mean():.0%} dos pares não têm. **Por que importa:** o modelo precisa saber que "
        "\"sem data\" é falta, não idade zero; a marca `tem_idade` já faz isso desde o ciclo 1.",
    "02_target_balance.png": f'**Equilíbrio do alvo.** "Não mesmo" é {_maioria:.1%} dos pares. **Por que importa:** '
        f'um palpite que sempre diz "não" acerta {_maioria:.1%}; por isso a régua é a regra do app, não o acerto geral.',
    "02_distributions.png": "**Distribuições das colunas numéricas.** A idade tem cauda longa (até décadas) e o "
        "cosseno vai de -0,2 a 1. **Por que importa:** a etapa 8 do ciclo 1 já trata isso (log da idade, padronização).",
    "02_correlations.png": "**Correlações com o alvo.** As maiores são as que o ciclo 1 já usa (cosseno, sobreposição). "
        "**Por que importa:** nenhuma acima de 0,95, o limite em que o pipeline desconfiaria de cola.",
}
_blocos = []
for _p in figuras_obrigatorias:
    _blocos.append(mo.image(src=str(_p), width=520))
    _blocos.append(mo.md(_explicacoes.get(_p.name, "")))
mo.vstack(_blocos)

In [ ]:
import matplotlib.pyplot as plt

# FIGURA PRÓPRIA 1: A RODADA 1, LADO A LADO. Três barras por rodada, todas "de cada 100 pares".
_por_rodada = pd.DataFrame({
    "a regra comparou com cuidado": 100 * tabela.groupby("rodada")["comparou"].mean(),
    'a regra diz "mesmo"': 100 * tabela.groupby("rodada")["regra"].apply(
        lambda s: s.isin(["mesmo_acontecimento", "detalhe_divergente"]).mean()),
    '"mesmo" de verdade (rótulo)': 100 * tabela.groupby("rodada")["mesmo"].mean(),
})
_fig, _ax = plt.subplots(figsize=(8, 3.8))
_por_rodada.plot.bar(ax=_ax, rot=0)                          # uma barra por coluna, agrupadas por rodada
_ax.set_ylabel("de cada 100 pares")
_ax.set_xlabel("rodada")
_ax.set_title("A rodada 1 é outro mundo? (rodadas 1 a 3, todas as fontes)")
_r = _por_rodada.round(1)
_explicacao = (
    f"Na rodada 1 a regra comparou com cuidado em {_r.iloc[0, 0]:.0f} de cada 100 pares, contra {_r.iloc[1, 0]:.0f} e "
    f"{_r.iloc[2, 0]:.0f} nas rodadas 2 e 3, e disse \"mesmo\" em {_r.iloc[0, 1]:.0f}, contra {_r.iloc[2, 1]:.0f} na rodada 3. "
    "Foi capturada com outra versão do app e com matérias escolhidas pelo grupo: um modelo que aprende com ela pode "
    "ligar \"a regra comparou\" a \"é o mesmo fato\" por um motivo que não existe nas outras rodadas."
)
figura_rodadas = guard.fig(2, "a_rodada_1_e_outro_mundo", _fig, _explicacao, state_dir=mo.notebook_dir())
mo.vstack([mo.image(src=str(figura_rodadas), width=620), mo.md(_explicacao)])

In [ ]:
# FIGURA PRÓPRIA 2: AS FONTES QUE ENGANAM. Cada ponto é uma fonte: semelhança com a notícia (vertical) contra a idade
# da fonte (horizontal, em escala log: 1 dia, 10 dias, 100 dias...). Sem data fica de fora do desenho e entra na conta.
_com_data = tabela[tabela["idade_dias"].notna()]
_x = np.log1p(_com_data["idade_dias"].clip(lower=0))       # log(1 + dias); -1 vira 0
_fig, _ax = plt.subplots(figsize=(8, 4.2))
for _valor, _cor, _nome, _tam in [(False, "#bbbbbb", "não é o mesmo fato", 4), (True, "#1f77b4", "mesmo fato", 6)]:
    _m = (_com_data["mesmo"] == _valor).to_numpy()
    _ax.scatter(_x[_m], _com_data["cosseno"][_m], s=_tam, c=_cor, label=_nome, alpha=0.6)
_ax.axhline(regua_cosseno, color="black", linestyle="--", linewidth=1)      # a régua do "parecer" da etapa 1
_ax.set_xticks(np.log1p([0, 1, 7, 30, 365, 3650]))
_ax.set_xticklabels(["0", "1 dia", "7", "30", "1 ano", "10 anos"])
_ax.set_xlabel("idade da fonte (dias, escala log)")
_ax.set_ylabel("semelhança com a notícia (cosseno)")
_ax.set_title("As fontes que enganam: parecidas (acima da linha) e que não são o mesmo fato (cinza)")
_ax.legend(loc="lower left", fontsize=8)

# A CONTA NOVA: entre as parecidas (cosseno na régua ou acima), quantas de cada 100 são o mesmo fato, por idade.
_parecidas = tabela[tabela["cosseno"] >= regua_cosseno]
_faixa = _parecidas["idade_dias"].map(
    lambda d: "sem data" if pd.isna(d) else "mesmo dia ou véspera" if d <= 1 else "2 a 30 dias" if d <= 30 else "mais de um mês")
acerto_das_parecidas = (pd.DataFrame({"parecidas": _faixa.value_counts(),
                                      "são o mesmo fato (de cada 100)": (100 * _parecidas.groupby(_faixa)["mesmo"].mean()).round(1)})
                        .reindex(["mesmo dia ou véspera", "2 a 30 dias", "mais de um mês", "sem data"]))
_a = acerto_das_parecidas["são o mesmo fato (de cada 100)"]
_explicacao = (
    f"Entre as fontes que se parecem com a notícia (cosseno de {regua_cosseno:.2f} ou mais), são de fato o mesmo fato "
    f"{_a['mesmo dia ou véspera']:.0f} de cada 100 no mesmo dia ou véspera, {_a['2 a 30 dias']:.0f} de 2 a 30 dias, "
    f"{_a['mais de um mês']:.0f} acima de um mês e {_a['sem data']:.0f} sem data. Os pontos cinza acima da linha "
    "são as fontes que enganam: a figura mostra em que idade elas moram."
)
figura_enganam = guard.fig(2, "as_fontes_que_enganam", _fig, _explicacao, state_dir=mo.notebook_dir())
mo.vstack([mo.image(src=str(figura_enganam), width=640), mo.md(_explicacao), acerto_das_parecidas])